# 📝 Instrucciones

## **Paso 1:** Busca un conjunto de datos

Investiga en las diferentes fuentes en línea sobre distintos datasets que podrías utilizar para entrenar un modelo. Puedes utilizar alguna API pública, el repositorio UCI para Machine Learning o el apartado de Kaggle de conjuntos de datos, entre otras muchas fuentes. Recuerda buscar un conjunto de datos simple ya que este no es el proyecto final del curso.

In [1]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import GridSearchCV
from joblib import dump

df = pd.read_csv('../data/raw/Gaming_Academic_Performance.csv', encoding='utf-8', sep=',')
df.head()

,student_id,age,gender,gaming_hours,study_hours,sleep_hours,attendance,gaming_genre,social_activity,device_usage,reaction_time_ms,addiction_score,stress_level,grades
0,1,22,Male,7.23,8.78,6.96,91.44,FPS,3.25,9.36,235.84,14.69,Low,86.459555
1,2,19,Male,0.07,8.72,7.63,63.63,Casual,1.02,3.21,328.71,2.47,Medium,98.230000
2,3,23,Female,1.73,9.56,4.40,83.26,Casual,3.46,5.56,313.61,4.73,High,90.560000
3,4,20,Female,6.62,1.68,7.83,75.04,RPG,1.46,11.78,241.84,14.54,Low,32.670000
4,5,22,Female,5.36,5.83,5.55,65.57,FPS,1.01,8.23,249.31,12.48,Low,58.710000


In [2]:
df.shape

(8000, 14)

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   student_id        8000 non-null   int64  
 1   age               8000 non-null   int64  
 2   gender            8000 non-null   str    
 3   gaming_hours      8000 non-null   float64
 4   study_hours       8000 non-null   float64
 5   sleep_hours       8000 non-null   float64
 6   attendance        8000 non-null   float64
 7   gaming_genre      8000 non-null   str    
 8   social_activity   8000 non-null   float64
 9   device_usage      8000 non-null   float64
 10  reaction_time_ms  8000 non-null   float64
 11  addiction_score   8000 non-null   float64
 12  stress_level      8000 non-null   str    
 13  grades            8000 non-null   float64
dtypes: float64(9), int64(2), str(3)
memory usage: 875.1 KB


**-Sin datos nulos**

**-Grades como variable objetivo. Será convertida a binaria para problema de clasificación**

In [4]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
student_id,8000.0,4000.500000,2309.545410,1.00,2000.750000,4000.500,6000.250000,8000.000000
age,8000.0,19.983625,2.587072,16.00,18.000000,20.000,22.000000,24.000000
gaming_hours,8000.0,4.085773,2.308801,0.00,2.130000,4.130,6.060000,8.000000
study_hours,8000.0,5.460581,2.575787,1.00,3.240000,5.460,7.660000,10.000000
sleep_hours,8000.0,6.493453,1.442656,4.00,5.240000,6.505,7.730000,9.000000
attendance,8000.0,79.886525,11.580419,60.00,69.780000,79.695,90.100000,100.000000
social_activity,8000.0,2.507790,1.441128,0.00,1.287500,2.500,3.760000,5.000000
device_usage,8000.0,7.586315,2.710035,1.10,5.560000,7.610,9.600000,13.950000
reaction_time_ms,8000.0,271.105839,29.440675,183.26,247.160000,270.475,294.690000,347.870000
addiction_score,8000.0,9.908492,5.035837,-4.51,5.920000,10.005,13.860000,23.160000


In [5]:
df["grades"].describe()

count    8000.000000
mean       66.180776
std        22.422024
min         0.000000
25%        49.879843
50%        67.070000
75%        83.992223
max       118.632936
Name: grades, dtype: float64

In [6]:
df = df.drop(columns=["student_id"])

In [7]:
df["passed"] = (df["grades"] >= 60).astype(int)
df = df.drop(columns=["grades"])
df["passed"].value_counts()

passed
1    4869
0    3131
Name: count, dtype: int64

In [8]:
df = pd.get_dummies(df, columns=["gender", "gaming_genre", "stress_level"])
bool_cols = df.select_dtypes(include="bool").columns
df[bool_cols] = df[bool_cols].astype(int)
df.shape

(8000, 19)

In [9]:
num_cols = ["age", "gaming_hours", "study_hours", "sleep_hours", "attendance",
            "social_activity", "device_usage", "reaction_time_ms", "addiction_score"]
scaler = MinMaxScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])
df.head()

,age,gaming_hours,study_hours,sleep_hours,attendance,social_activity,device_usage,reaction_time_ms,addiction_score,passed,gender_Female,gender_Male,gender_Other,gaming_genre_Casual,gaming_genre_FPS,gaming_genre_RPG,stress_level_High,stress_level_Low,stress_level_Medium
0,0.750,0.90375,0.864444,0.592,0.78600,0.650,0.642802,0.319422,0.693892,1,0,1,0,0,1,0,0,1,0
1,0.375,0.00875,0.857778,0.726,0.09075,0.204,0.164202,0.883604,0.252259,1,0,1,0,1,0,0,0,0,1
2,0.875,0.21625,0.951111,0.080,0.58150,0.692,0.347082,0.791872,0.333936,1,1,0,0,1,0,0,1,0,0
3,0.500,0.82750,0.075556,0.766,0.37600,0.292,0.831128,0.355871,0.688471,0,1,0,0,0,0,1,0,1,0
4,0.750,0.67000,0.536667,0.310,0.13925,0.202,0.554864,0.401251,0.614022,0,1,0,0,0,1,0,0,1,0


## **Paso 2:** Desarrolla un modelo
Una vez hayas encontrado tu conjunto de datos ideal, analízalo y entrena un modelo. Optimízalo si fuera necesario.

In [10]:
X = df.drop(columns=["passed"])
y = df["passed"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train: {X_train.shape}")
print(f"Test:  {X_test.shape}")

Train: (6400, 18)
Test:  (1600, 18)


In [11]:
X_train.shape, X_test.shape

((6400, 18), (1600, 18))

In [12]:
model = SVC(random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, target_names=["Suspendido", "Aprobado"]))

Accuracy: 0.9244
              precision    recall  f1-score   support

  Suspendido       0.92      0.89      0.90       631
    Aprobado       0.93      0.95      0.94       969

    accuracy                           0.92      1600
   macro avg       0.92      0.92      0.92      1600
weighted avg       0.92      0.92      0.92      1600



**El modelo identifica correctamente el 89% de los suspendidos y el 95% de los aprobados, lo que indica que tiene más dificultad distinguiendo casos límites de suspenso que aprobados.**

In [13]:
param_grid = {
    "kernel": ["linear", "rbf"],
    "C": [0.1, 1, 10],
    "gamma": ["scale", "auto"]
}

grid_search = GridSearchCV(
    estimator=SVC(random_state=42),
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f"Mejores parámetros: {grid_search.best_params_}")
print(f"Mejor F1 en CV: {grid_search.best_score_:.4f}")

best_model = grid_search.best_estimator_
y_pred_opt = best_model.predict(X_test)

print(f"\nAccuracy: {accuracy_score(y_test, y_pred_opt):.4f}")
print(classification_report(y_test, y_pred_opt, target_names=["Suspendido", "Aprobado"]))

Fitting 5 folds for each of 12 candidates, totalling 60 fits
Mejores parámetros: {'C': 1, 'gamma': 'scale', 'kernel': 'linear'}
Mejor F1 en CV: 0.9421

Accuracy: 0.9263
              precision    recall  f1-score   support

  Suspendido       0.92      0.89      0.90       631
    Aprobado       0.93      0.95      0.94       969

    accuracy                           0.93      1600
   macro avg       0.93      0.92      0.92      1600
weighted avg       0.93      0.93      0.93      1600



**El accuracy subió apenas 0.2 puntos, lo que indica que el modelo base ya estaba bien ajustado para estos datos.**

In [ ]:
dump(best_model, "..gaming_vs_academic/models/svm_gaming_academic_linear_42.joblib")
dump(scaler, "..gaming_vs_academic/models/scaler_gaming_academic.joblib")

['../models/scaler_gaming_academic.joblib']

## Modelo con estimación de probabilidad

El modelo anterior solo devuelve predicciones binarias (0/1). Para poder mostrar 
una probabilidad individual por predicción en la aplicación web, reentrenamos 
activando `probability=True`, que aplica calibración de Platt internamente: 
transforma la distancia de cada punto al hiperplano en un valor entre 0 y 1 
interpretable como probabilidad.

In [15]:
grid_search_prob = GridSearchCV(
    estimator=SVC(random_state=42, probability=True),
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_search_prob.fit(X_train, y_train)

print(f"Mejores parámetros: {grid_search_prob.best_params_}")
print(f"Mejor F1 en CV: {grid_search_prob.best_score_:.4f}")

best_model_prob = grid_search_prob.best_estimator_
y_pred_prob = best_model_prob.predict(X_test)

print(f"\nAccuracy: {accuracy_score(y_test, y_pred_prob):.4f}")
print(classification_report(y_test, y_pred_prob, target_names=["Suspendido", "Aprobado"]))

Fitting 5 folds for each of 12 candidates, totalling 60 fits
Mejores parámetros: {'C': 1, 'gamma': 'scale', 'kernel': 'linear'}
Mejor F1 en CV: 0.9421

Accuracy: 0.9263
              precision    recall  f1-score   support

  Suspendido       0.92      0.89      0.90       631
    Aprobado       0.93      0.95      0.94       969

    accuracy                           0.93      1600
   macro avg       0.93      0.92      0.92      1600
weighted avg       0.93      0.93      0.93      1600



**Se obtuvieron los mismos resultados, así que la calibración no afectó la precisión.**

In [16]:
dump(best_model_prob, "../models/gaming_vs_academic/svm_gaming_academic_linear_42.joblib")

['../models/gaming_vs_academic/svm_gaming_academic_linear_42.joblib']

## **Paso 3:** Desarrolla una aplicación web usando Flask

Con los conocimientos adquiridos en este módulo, desarrolla una interfaz para poder utilizar el modelo. Dale el estilo que más te convenga y anota los recursos externos que hayas utilizado para el desarrollo.

## **Paso 4:** Integra el modelo y la aplicación en Render
Crea un servicio gratuito en Render e integra el trabajo que has hecho para poder desplegar la aplicación web en línea. No olvides de incluir el enlace al servicio en tu repositorio.